In [1]:
import os

from PIL import Image
from dataclasses import dataclass
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import TensorDataset, DataLoader, SubsetRandomSampler, WeightedRandomSampler
from torchmetrics.classification import (
    BinaryAccuracy,
    BinaryF1Score,
    BinaryConfusionMatrix,
    BinaryMatthewsCorrCoef
)
import plotly.express as px

In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# CNN

In [3]:
@dataclass
class CNNConfig:
    layers: list[nn.Module]

In [4]:
class CNN(nn.Module):
    def __init__(self, config: CNNConfig):
        super().__init__()
        self.model = nn.Sequential(*config.layers)

    def forward(self, xs):
        return self.model(xs)



In [5]:
def create_model(config: CNNConfig) -> CNN:
    return CNN(config).to(device)

# Training

In [6]:
@dataclass
class Hyperparameters:
    epochs: int
    lr: float
    batch_size: int
    patience: int

In [7]:
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.should_stop = True

In [8]:
@dataclass
class Results:
    train_losses: list[float]
    test_losses: list[float]
    accuracies: list[float]
    f1s: list[float]
    sensitivities: list[float]
    specificities: list[float]
    mccs: list[float]

In [10]:
def train_test(model: CNN, H: Hyperparameters,
               X_train, y_train,
               X_val, y_val) -> Results:

    results = Results([], [], [], [], [], [], [])


    y_np = y_train.cpu().numpy().reshape(-1).astype(np.int64)

    class_counts = np.bincount(y_np)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[y_np]

    sampler = WeightedRandomSampler(
        torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
    )

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=H.lr)
    early_stop = EarlyStopping(H.patience)

    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=H.batch_size, sampler=sampler)

    accuracy_fn = BinaryAccuracy().to(device)
    f1_fn = BinaryF1Score().to(device)
    cm_fn = BinaryConfusionMatrix().to(device)
    mcc_fn = BinaryMatthewsCorrCoef().to(device)

    for epoch in range(H.epochs):
        model.train()
        total_train_loss = 0.0

        for X_batch, y_batch in train_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        results.train_losses.append(total_train_loss / len(train_loader))

        model.eval()
        with torch.no_grad():
            logits = model(X_val)
            val_loss = criterion(logits, y_val)
            results.test_losses.append(val_loss.item())

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

            accuracy = accuracy_fn(preds, y_val)
            results.accuracies.append(accuracy.item())

            f1 = f1_fn(preds, y_val)
            results.f1s.append(f1.item())

            cm = cm_fn(preds, y_val)
            tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]

            sensitivity = (tp / (tp + fn)).item() if tp + fn > 0 else 0
            specificity = (tn / (tn + fp)).item() if tn + fp > 0 else 0

            results.sensitivities.append(sensitivity)
            results.specificities.append(specificity)

            mcc = mcc_fn(preds, y_val)
            results.mccs.append(mcc.item())

        early_stop.step(val_loss, model)
        if early_stop.should_stop:
            print(f"\n⛔ Early stopping at epoch {epoch+1}")
            break

        print(f"Epoch {epoch+1}/{H.epochs}")

    del optimizer
    del criterion
    if device == "cuda":
        with torch.no_grad():
            torch.cuda.empty_cache()

    return results


In [11]:
def losses_plot(train_losses, test_losses):
    df = pd.DataFrame({
        "Epochs": list(range(len(train_losses))),
        "Train Loss": train_losses,
        "Test Loss": test_losses,
    })

    df_melted = df.melt(id_vars="Epochs", value_vars=["Train Loss", "Test Loss"],
                        var_name="Metric", value_name="Loss")

    fig = px.line(
        df_melted,
        x="Epochs",
        y="Loss",
        color="Metric",
        markers=True,
        title=f"Train & Test Loss vs Epochs"
    )
    fig.show()

In [12]:
def metrics_plot(accuracies, f1s, sensitivities, specificities, mccs):
    df = pd.DataFrame({
        "Epochs": list(range(len(accuracies))),
        "Accuracy": accuracies,
        "F1": f1s,
        "Sensitivity": sensitivities,
        "Specificity": specificities,
        "MCC": mccs,
    })

    df_melted = df.melt(id_vars="Epochs", value_vars=["Accuracy", "F1", "Sensitivity", "Specificity", "MCC"],
                        var_name="Metric", value_name="Score")

    fig = px.line(
        df_melted,
        x="Epochs",
        y="Score",
        color="Metric",
        markers=True,
        title="Val scores vs Epochs"
    )
    fig.show()

In [13]:
def confusion_matrix_plot(model: CNN, X_test, y_test):
    # Ensure model and data are on CPU
    model = model.to("cpu")
    X_test = X_test.cpu()
    y_test = y_test.cpu()

    cm_fn = BinaryConfusionMatrix().to("cpu")

    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int()

        cm = cm_fn(preds, y_test)

    fig = px.imshow(cm.numpy(), text_auto=True,
                    labels=dict(x="Predicted", y="Actual"))
    fig.show()


# Dataset

In [14]:
image_size = (224, 224)

In [15]:
df = pd.read_pickle("../data/skin-lesion-data/binary.pickle")

In [16]:
def load_images(df, root_dir, image_size=(224,224)):
    X = []
    y = []

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])

    for _, row in df.iterrows():
        path = os.path.join(root_dir, row["img_id"])
        img = Image.open(path).convert("RGB")
        img = img.resize(image_size)

        arr = np.array(img).astype(np.float32) / 255.0

        arr = (arr - mean) / std

        X.append(arr)
        y.append(int(row["malignant"]))

    return np.array(X), np.array(y)


In [17]:
X, y = load_images(df, "../data/skin-lesion-data/images/train")
y = y[:, np.newaxis]

In [18]:
X.shape, y.shape

((773, 224, 224, 3), (773, 1))

In [19]:
y[:5]

array([[1],
       [1],
       [1],
       [1],
       [0]])

In [20]:
def numpy_to_torch(x):
    return torch.from_numpy(x).float().to(device)

In [21]:
def train_val_test(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
    X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
    X_train = numpy_to_torch(X_train)
    X_val = numpy_to_torch(X_val)
    X_test = numpy_to_torch(X_test)
    y_train = numpy_to_torch(y_train)
    y_val = numpy_to_torch(y_val)
    y_test = numpy_to_torch(y_test)
    return X_train, X_val, X_test, y_train, y_val, y_test

In [22]:
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test(X, y)

# X_train, X_val and X_test have shape (batch, H, W, C)
# PyTorch Conv2d expects the shape (batch, C, H, W)
X_train = X_train.permute(0, 3, 1, 2)
X_val = X_val.permute(0, 3, 1, 2)
X_test = X_test.permute(0, 3, 1, 2)



In [23]:
X_train.shape, y_train.shape

(torch.Size([463, 3, 224, 224]), torch.Size([463, 1]))

In [24]:
X_val.shape, y_val.shape

(torch.Size([155, 3, 224, 224]), torch.Size([155, 1]))

In [25]:
X_test.shape, y_test.shape

(torch.Size([155, 3, 224, 224]), torch.Size([155, 1]))

In [29]:
model = create_model(
    CNNConfig([
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),  # 112x112

    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),  # 56x56

    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28

    nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14

    nn.Flatten(),  # flatten for fully connected layers
    nn.Linear(128 * 14 * 14, 512),
    nn.ReLU(),
    nn.Linear(512, 1)  # example: 10 classes
    ])
)

In [27]:
results = train_test(model, Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10), X_train, y_train, X_val, y_val)

Epoch 1/100
Epoch 2/100
Epoch 3/100
Epoch 4/100
Epoch 5/100
Epoch 6/100
Epoch 7/100
Epoch 8/100
Epoch 9/100
Epoch 10/100
Epoch 11/100
Epoch 12/100
Epoch 13/100
Epoch 14/100
Epoch 15/100

⛔ Early stopping at epoch 16


In [28]:
losses_plot(results.train_losses, results.test_losses)

In [29]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [30]:
confusion_matrix_plot(model, X_val, y_val)

# Hyperparameter finetuning

In [ ]:
configs = {
    "baseline": nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 224 -> 112
        nn.Conv2d(16, 32, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 112 -> 56
        nn.AdaptiveAvgPool2d((7,7)),  # 56 -> 7
        nn.Flatten(),
        nn.Linear(32*7*7, 512),
        nn.ReLU(),
        nn.Linear(512, 1)
    ),

    "shallow & narrow": nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 224 -> 112
        nn.AdaptiveAvgPool2d((7,7)),  # 112 -> 7
        nn.Flatten(),
        nn.Linear(16*7*7, 1)
    ),

    "deep & narrow": nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(16, 16, 3, padding=1),
        nn.Tanh(),
        nn.MaxPool2d(2),  # 224 -> 112
        nn.Conv2d(16, 32, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(32, 32, 3, padding=1),
        nn.Tanh(),
        nn.MaxPool2d(2),  # 112 -> 56
        nn.AdaptiveAvgPool2d((7,7)),  # 56 -> 7
        nn.Flatten(),
        nn.Linear(32*7*7, 1)
    ),

    "shallow & wide": nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),  # 224 -> 112
        nn.AdaptiveAvgPool2d((7,7)),  # 112 -> 7
        nn.Flatten(),
        nn.Linear(32*7*7, 1)
    ),

    "deep & wide": nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(32, 64, 3, padding=1),
        nn.Tanh(),
        nn.MaxPool2d(2),  # 224 -> 112
        nn.Conv2d(64, 128, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(128, 128, 3, padding=1),
        nn.Tanh(),
        nn.MaxPool2d(2),  # 112 -> 56
        nn.AdaptiveAvgPool2d((7,7)),  # 56 -> 7
        nn.Flatten(),
        nn.Linear(128*7*7, 1)
    ),
    "LeNet-5": nn.Sequential(
        nn.Conv2d(3, 6, kernel_size=5, padding=2),
        nn.Tanh(),
        nn.AvgPool2d(2, 2),
        nn.Conv2d(6, 16, kernel_size=5),
        nn.Tanh(),
        nn.AvgPool2d(2, 2),
        nn.Flatten(),
        nn.Linear(16 * 54 * 54, 120),
        nn.Tanh(),
        nn.Linear(120, 84),
        nn.Tanh(),
        nn.Linear(84, 1)
    ),
    "Giri": nn.Sequential(
        # ------- Block 1 -------
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # 224 -> 112

        # ------- Block 2 -------
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # 112 -> 56

        # ------- Block 3 -------
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # 56 -> 28

        # ------- Block 4 -------
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),  # 28 -> 14

        # ------- Flatten -------
        nn.Flatten(),

        # ------- Fully Connected -------
        nn.Linear(256 * 14 * 14, 512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(128, 1)
    )
}


In [32]:
all_results = {}

for name, config in configs.items():
    print(f"Training model: {name}")

    model = create_model(CNNConfig(config))
    results = train_test(
        model,
        Hyperparameters(epochs=100, lr=1e-3, batch_size=24, patience=10),
        X_train,
        y_train,
        X_val,
        y_val
    )

    all_results[name] = results

Training model: baseline
Epoch 1/100
Epoch 2/100
Epoch 3/100
Epoch 4/100
Epoch 5/100
Epoch 6/100
Epoch 7/100
Epoch 8/100
Epoch 9/100
Epoch 10/100
Epoch 11/100
Epoch 12/100
Epoch 13/100
Epoch 14/100
Epoch 15/100
Epoch 16/100
Epoch 17/100
Epoch 18/100
Epoch 19/100
Epoch 20/100

⛔ Early stopping at epoch 21
Training model: shallow & narrow
Epoch 1/100
Epoch 2/100
Epoch 3/100
Epoch 4/100
Epoch 5/100
Epoch 6/100
Epoch 7/100
Epoch 8/100
Epoch 9/100
Epoch 10/100
Epoch 11/100
Epoch 12/100
Epoch 13/100
Epoch 14/100
Epoch 15/100
Epoch 16/100
Epoch 17/100
Epoch 18/100
Epoch 19/100
Epoch 20/100
Epoch 21/100
Epoch 22/100
Epoch 23/100
Epoch 24/100
Epoch 25/100
Epoch 26/100
Epoch 27/100

⛔ Early stopping at epoch 28
Training model: deep & narrow
Epoch 1/100
Epoch 2/100
Epoch 3/100
Epoch 4/100
Epoch 5/100
Epoch 6/100
Epoch 7/100
Epoch 8/100
Epoch 9/100
Epoch 10/100
Epoch 11/100
Epoch 12/100
Epoch 13/100
Epoch 14/100

⛔ Early stopping at epoch 15
Training model: shallow & wide
Epoch 1/100
Epoch 2/100
E

In [33]:
summary = pd.DataFrame([
    {
        "model": name,
        "final_train_loss": res.train_losses[-1],
        "final_val_loss": res.test_losses[-1],
        "final_accuracy": res.accuracies[-1],
        "final_f1": res.f1s[-1],
        "final_sensitivity": res.sensitivities[-1],
        "final_specificity": res.specificities[-1],
        "final_mcc": res.mccs[-1],
    }
    for name, res in all_results.items()
])
summary = summary.set_index("model")

In [34]:
summary

,final_train_loss,final_val_loss,final_accuracy,final_f1,final_sensitivity,final_specificity,final_mcc
model,,,,,,,
baseline,0.257818,0.725573,0.748387,0.800000,0.821053,0.633333,0.462574
shallow & narrow,0.491496,0.541095,0.709677,0.754098,0.726316,0.683333,0.402783
deep & narrow,0.420491,0.569687,0.696774,0.743169,0.715789,0.666667,0.376045
shallow & wide,0.488616,0.550333,0.741935,0.782609,0.757895,0.716667,0.467480
deep & wide,0.456116,0.558362,0.729032,0.771739,0.747368,0.700000,0.440693
LeNet-5,0.628111,0.697042,0.580645,0.615385,0.547368,0.633333,0.176214
Giri,0.482107,0.518404,0.748387,0.774566,0.705263,0.816667,0.508460


In [35]:
fig = px.imshow(summary[["final_train_loss", "final_val_loss"]], text_auto=True)
fig.show()

In [36]:
fig = px.imshow(summary.drop(columns=["final_train_loss", "final_val_loss"]), text_auto=True)
fig.show()

In [ ]:
model = create_model(
    CNNConfig(
        [
            # ------- Block 1 -------
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 224 -> 112

            # ------- Block 2 -------
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112 -> 56

            # ------- Block 3 -------
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 56 -> 28

            # ------- Block 4 -------
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28 -> 14

            # ------- Flatten -------
            nn.Flatten(),

            # ------- Fully Connected -------
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 1)
        ]
    )
)

In [33]:
results = train_test(model, Hyperparameters(epochs=250, lr=1e-4, batch_size=24, patience=5), X_train, y_train, X_test, y_test)

Epoch 1/250
Epoch 2/250
Epoch 3/250
Epoch 4/250
Epoch 5/250
Epoch 6/250
Epoch 7/250
Epoch 8/250
Epoch 9/250
Epoch 10/250
Epoch 11/250
Epoch 12/250
Epoch 13/250

⛔ Early stopping at epoch 14


In [34]:
losses_plot(results.train_losses, results.test_losses)

In [35]:
metrics_plot(results.accuracies, results.f1s, results.sensitivities, results.specificities, results.mccs)

In [36]:
confusion_matrix_plot(model, X_test, y_test)